# Deep Learning Assignment 2
**Name:** Deviprasad  
**USN:** NNM23IS047  
**Section & Semester:** 6th A  
**Course:** Deep Learning (IS3332-1)  
**Submitted To:** Dr. Jason Elroy Martis

---
## Contents
- **Part A** — Image Classification using Custom CNN and ResNet18 (MNIST)
- **Part B** — Sentiment Classification using RNN, LSTM, GRU (IMDB)

---
# PART A — Image Classification using CNN (MNIST)
---

In [ ]:
# ── Install & Import Libraries ────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ── Load MNIST Dataset ────────────────────────────────────────────────────────
transform_train = transforms.Compose([
    transforms.Resize((32, 32)),           # Resize for ResNet18 compatibility
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))   # Single channel normalisation
])

transform_test = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform_train)
test_dataset  = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=transform_test)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=64, shuffle=False)

classes = [str(i) for i in range(10)]
print(f'Train size: {len(train_dataset)} | Test size: {len(test_dataset)}')

In [ ]:
# ── Visualise Sample Images ───────────────────────────────────────────────────
dataiter = iter(train_loader)
images, labels = next(dataiter)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img = images[i].squeeze().numpy()
    img = (img * 0.5) + 0.5  # Unnormalise
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Digit: {labels[i].item()}', fontsize=8)
    ax.axis('off')
plt.suptitle('Sample MNIST Images', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Custom CNN Architecture ───────────────────────────────────────────────────
class CustomCNN(nn.Module):
    def __init__(self):
        super(CustomCNN, self).__init__()

        # Conv Block 1
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)   # 32x32 -> 16x16
        )

        # Conv Block 2
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)   # 16x16 -> 8x8
        )

        # Fully Connected Layers
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.fc(x)
        return x

model_cnn = CustomCNN().to(device)
print(model_cnn)
total_params = sum(p.numel() for p in model_cnn.parameters())
print(f'\nTotal Parameters: {total_params:,}')

In [ ]:
# ── Train Custom CNN ──────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer_cnn = optim.Adam(model_cnn.parameters(), lr=0.001)

NUM_EPOCHS = 10
cnn_train_loss, cnn_train_acc, cnn_val_acc = [], [], []

for epoch in range(NUM_EPOCHS):
    model_cnn.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_cnn.zero_grad()
        outputs = model_cnn(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_cnn.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_acc  = 100. * correct / total

    # Validation
    model_cnn.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_cnn(images)
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total += labels.size(0)
    val_acc = 100. * val_correct / val_total

    cnn_train_loss.append(train_loss)
    cnn_train_acc.append(train_acc)
    cnn_val_acc.append(val_acc)

    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] | Loss: {train_loss:.4f} | '
          f'Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%')

print(f'\nFinal Test Accuracy (Custom CNN): {cnn_val_acc[-1]:.2f}%')

In [ ]:
# ── Plot CNN Training Curves ──────────────────────────────────────────────────
epochs_range = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, cnn_train_loss, 'g-o', label='Training Loss', linewidth=2)
axes[0].set_title('Custom CNN — Training Loss', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, cnn_train_acc, 'g-o', label='Train Accuracy', linewidth=2)
axes[1].plot(epochs_range, cnn_val_acc,   'b-^', label='Val Accuracy',   linewidth=2)
axes[1].set_title('Custom CNN — Accuracy', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion Matrix — Custom CNN ─────────────────────────────────────────────
model_cnn.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model_cnn(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix — Custom CNN (MNIST)', fontweight='bold', fontsize=13)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(all_labels, all_preds, target_names=classes))

In [ ]:
# ── Transfer Learning — ResNet18 ─────────────────────────────────────────────
# Modify ResNet18: accepts 1-channel input, outputs 10 classes
model_resnet = models.resnet18(pretrained=True)

# Modify first conv layer: 3 channels -> 1 channel
model_resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

# Freeze all pretrained layers
for param in model_resnet.parameters():
    param.requires_grad = False

# Only train the final classification head
model_resnet.fc = nn.Linear(model_resnet.fc.in_features, 10)
for param in model_resnet.fc.parameters():
    param.requires_grad = True
# Also train the modified first conv layer
for param in model_resnet.conv1.parameters():
    param.requires_grad = True

model_resnet = model_resnet.to(device)
trainable = sum(p.numel() for p in model_resnet.parameters() if p.requires_grad)
print(f'Trainable Parameters: {trainable:,}')

In [ ]:
# ── Train ResNet18 ────────────────────────────────────────────────────────────
optimizer_resnet = optim.Adam(
    filter(lambda p: p.requires_grad, model_resnet.parameters()), lr=0.001)

RESNET_EPOCHS = 5
resnet_train_loss, resnet_val_acc = [], []

for epoch in range(RESNET_EPOCHS):
    model_resnet.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_resnet.zero_grad()
        outputs = model_resnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_resnet.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader)

    # Validation
    model_resnet.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_resnet(images)
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total += labels.size(0)
    val_acc = 100. * val_correct / val_total

    resnet_train_loss.append(train_loss)
    resnet_val_acc.append(val_acc)
    print(f'Epoch [{epoch+1}/{RESNET_EPOCHS}] | Loss: {train_loss:.4f} | Val Acc: {val_acc:.2f}%')

print(f'\nFinal Test Accuracy (ResNet18): {resnet_val_acc[-1]:.2f}%')

In [ ]:
# ── Compare CNN vs ResNet18 ───────────────────────────────────────────────────
print('='*45)
print(f'  Model Comparison — MNIST Classification')
print('='*45)
print(f'  Custom CNN   : {cnn_val_acc[-1]:.2f}%')
print(f'  ResNet18     : {resnet_val_acc[-1]:.2f}%')
print('='*45)

# Bar chart comparison
plt.figure(figsize=(6, 4))
bars = plt.bar(['Custom CNN', 'ResNet18'],
               [cnn_val_acc[-1], resnet_val_acc[-1]],
               color=['#2d6a4f', '#52b788'], width=0.4, edgecolor='white')
plt.ylim([80, 100])
plt.ylabel('Test Accuracy (%)', fontsize=11)
plt.title('CNN vs ResNet18 — MNIST', fontweight='bold', fontsize=13)
plt.grid(axis='y', alpha=0.3)
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.2,
             f'{bar.get_height():.2f}%',
             ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

---
# PART B — Sentiment Classification using RNN, LSTM, GRU (IMDB)
---

In [ ]:
# ── Install TensorFlow dependencies ──────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import numpy as np

print(f'TensorFlow version: {tf.__version__}')

In [ ]:
# ── Load and Preprocess IMDB Dataset ─────────────────────────────────────────
VOCAB_SIZE   = 10000   # Top 10,000 most frequent words
MAX_LEN      = 200     # Pad/truncate all sequences to 200 tokens
EMBED_DIM    = 128     # Embedding dimension
HIDDEN_UNITS = 128     # Recurrent hidden units
BATCH_SIZE   = 64
EPOCHS       = 5

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

X_train = pad_sequences(X_train, maxlen=MAX_LEN, padding='post', truncating='post')
X_test  = pad_sequences(X_test,  maxlen=MAX_LEN, padding='post', truncating='post')

print(f'Train shape: {X_train.shape} | Test shape: {X_test.shape}')
print(f'Sample label — 0=Negative, 1=Positive: {y_train[:5]}')

In [ ]:
# ── Helper: Build Recurrent Model ─────────────────────────────────────────────
def build_model(model_type='LSTM'):
    model = Sequential(name=model_type)
    model.add(Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM, input_length=MAX_LEN))

    if model_type == 'RNN':
        model.add(SimpleRNN(HIDDEN_UNITS, return_sequences=False))
    elif model_type == 'LSTM':
        model.add(LSTM(HIDDEN_UNITS, return_sequences=False))
    elif model_type == 'GRU':
        model.add(GRU(HIDDEN_UNITS, return_sequences=False))

    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Early stopping to prevent overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

print('Model builder ready!')

In [ ]:
# ── Train Simple RNN ──────────────────────────────────────────────────────────
print('Training Simple RNN...')
model_rnn = build_model('RNN')
model_rnn.summary()

history_rnn = model_rnn.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)
rnn_loss, rnn_acc = model_rnn.evaluate(X_test, y_test, verbose=0)
print(f'\nSimple RNN — Test Accuracy: {rnn_acc*100:.2f}% | Test Loss: {rnn_loss:.4f}')

In [ ]:
# ── Train LSTM ────────────────────────────────────────────────────────────────
print('Training LSTM...')
model_lstm = build_model('LSTM')
model_lstm.summary()

history_lstm = model_lstm.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)
lstm_loss, lstm_acc = model_lstm.evaluate(X_test, y_test, verbose=0)
print(f'\nLSTM — Test Accuracy: {lstm_acc*100:.2f}% | Test Loss: {lstm_loss:.4f}')

In [ ]:
# ── Train GRU ─────────────────────────────────────────────────────────────────
print('Training GRU...')
model_gru = build_model('GRU')
model_gru.summary()

history_gru = model_gru.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)
gru_loss, gru_acc = model_gru.evaluate(X_test, y_test, verbose=0)
print(f'\nGRU — Test Accuracy: {gru_acc*100:.2f}% | Test Loss: {gru_loss:.4f}')

In [ ]:
# ── Plot Validation Accuracy Comparison ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy
axes[0].plot(history_rnn.history['val_accuracy'],  'r-o', label='Simple RNN', linewidth=2)
axes[0].plot(history_lstm.history['val_accuracy'], 'g-s', label='LSTM',       linewidth=2)
axes[0].plot(history_gru.history['val_accuracy'],  'b-^', label='GRU',        linewidth=2)
axes[0].set_title('Validation Accuracy — RNN Models', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history_rnn.history['val_loss'],  'r-o', label='Simple RNN', linewidth=2)
axes[1].plot(history_lstm.history['val_loss'], 'g-s', label='LSTM',       linewidth=2)
axes[1].plot(history_gru.history['val_loss'],  'b-^', label='GRU',        linewidth=2)
axes[1].set_title('Validation Loss — RNN Models', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Final Results Summary ─────────────────────────────────────────────────────
print('='*50)
print('         FINAL RESULTS SUMMARY')
print('='*50)
print(f'  PART A — Image Classification (MNIST)')
print(f'  Custom CNN   : {cnn_val_acc[-1]:.2f}%')
print(f'  ResNet18     : {resnet_val_acc[-1]:.2f}%')
print('-'*50)
print(f'  PART B — Sentiment Classification (IMDB)')
print(f'  Simple RNN   : {rnn_acc*100:.2f}%')
print(f'  LSTM         : {lstm_acc*100:.2f}%')
print(f'  GRU          : {gru_acc*100:.2f}%')
print('='*50)

# Bar chart — all models
fig, ax = plt.subplots(figsize=(9, 4))
model_names = ['Custom CNN', 'ResNet18', 'Simple RNN', 'LSTM', 'GRU']
accuracies  = [cnn_val_acc[-1], resnet_val_acc[-1],
               rnn_acc*100, lstm_acc*100, gru_acc*100]
colors = ['#2d6a4f','#52b788','#e76f51','#e76f51','#e76f51']
bars = ax.bar(model_names, accuracies, color=colors, width=0.5, edgecolor='white')
ax.set_ylim([70, 102])
ax.set_ylabel('Test Accuracy (%)', fontsize=11)
ax.set_title('Overall Model Accuracy Comparison', fontweight='bold', fontsize=13)
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.3,
            f'{val:.2f}%', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Test with Custom Review ───────────────────────────────────────────────────
word_index = imdb.get_word_index()

def predict_sentiment(review_text, model):
    words = review_text.lower().split()
    seq   = [word_index.get(w, 2) + 3 for w in words]
    seq   = pad_sequences([seq], maxlen=MAX_LEN, padding='post', truncating='post')
    pred  = model.predict(seq, verbose=0)[0][0]
    sentiment = 'POSITIVE 😊' if pred > 0.5 else 'NEGATIVE 😞'
    print(f'Review   : "{review_text}"')
    print(f'Sentiment: {sentiment} (confidence: {pred:.2f})')
    print()

# Test examples
predict_sentiment("This movie was absolutely fantastic and brilliant", model_gru)
predict_sentiment("The film was boring and a complete waste of time", model_gru)
predict_sentiment("An average movie with some good moments", model_gru)